# 04 — Topic modeling với TF-IDF + NMF (phiên bản làm sạch)

Phiên bản này loại tiền tố `Title:`/`Post:` và URL trước khi fit để giảm topic nhiễu. Fit trên mẫu tối đa 50.000 bài, sau đó gán chủ đề toàn bộ dữ liệu theo batch. Kết quả V2 được lưu riêng để giữ lại kết quả cũ và so sánh.


In [ ]:
%pip -q install pandas pyarrow scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import html
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/RedditProject')
DATA_DIR = PROJECT_DIR / 'data/processed'
SCORED_PATH = DATA_DIR / 'reddit_posts_sentiment.parquet'
assert SCORED_PATH.exists(), f'Không thấy {SCORED_PATH}; chạy 03_sentiment_trends.ipynb trước.'
N_TOPICS = 12
FIT_SAMPLE = 50_000
BATCH_SIZE = 10_000
RANDOM_STATE = 42
# Các từ này đến từ cấu trúc ghép title/body, URL hoặc boilerplate; không mang chủ đề ổn định.
CUSTOM_STOP_WORDS = {
    'title','post','https','http','www','com','reddit','redd','preview','png','webp',
    'amp','width','height','format','auto','utm','share','url','link','imgur',
    'like','just','know','want','using','use','really','think','thing','things'
}
STOP_WORDS = list(ENGLISH_STOP_WORDS.union(CUSTOM_STOP_WORDS))

def clean_reddit_text(value):
    text = html.unescape(str(value or ''))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'(?i)\b(title|post)\s*:\s*', ' ', text)
    text = re.sub(r'(?i)\b(amp|nbsp|quot|lt|gt)\b', ' ', text)
    text = re.sub(r'[^\w+#.-]+', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()


In [ ]:
# Fit TF-IDF/NMF trên mẫu ngẫu nhiên cố định để giới hạn RAM.
base = pd.read_parquet(SCORED_PATH, columns=['id','subreddit','created_at_utc','text','sentiment_label'])
fit_df = base.sample(n=min(FIT_SAMPLE, len(base)), random_state=RANDOM_STATE).copy()
texts = fit_df['text'].map(clean_reddit_text).tolist()
vectorizer = TfidfVectorizer(
    max_features=40_000, min_df=10, max_df=0.80, stop_words=STOP_WORDS,
    ngram_range=(1,2), strip_accents='unicode', sublinear_tf=True
)
X_fit = vectorizer.fit_transform(texts)
nmf = NMF(n_components=N_TOPICS, init='nndsvda', random_state=RANDOM_STATE, max_iter=250, alpha_W=0.00005, l1_ratio=0.1)
W_fit = nmf.fit_transform(X_fit)
fit_df['topic_id'] = W_fit.argmax(axis=1)
fit_share = fit_df['topic_id'].value_counts(normalize=True).sort_index().mul(100)
print('Fit rows:', len(fit_df), '| vocabulary:', len(vectorizer.get_feature_names_out()), '| topics:', N_TOPICS)
print('Topic share trong mẫu fit (%):')
display(fit_share.rename('percent').to_frame())
if fit_share.max() > 50:
    print('CẢNH BÁO: Một topic vẫn chiếm hơn 50% mẫu fit; cần xem lại từ khóa/bài mẫu trước khi kết luận topic prevalence.')


In [ ]:
# In từ khóa nổi bật và bài đại diện; đối chiếu chúng trước khi đặt tên topic.
terms = vectorizer.get_feature_names_out()
topic_rows = []
for k, weights in enumerate(nmf.components_):
    top_idx = weights.argsort()[-12:][::-1]
    topic_rows.append({'topic_id': k, 'top_terms': ', '.join(terms[top_idx])})
topics = pd.DataFrame(topic_rows)
display(topics)
representatives = []
for k in range(N_TOPICS):
    idx = np.flatnonzero(fit_df['topic_id'].to_numpy() == k)
    if len(idx):
        strengths = W_fit[idx, k]
        chosen = idx[np.argsort(strengths)[-5:][::-1]]
        representatives.extend(fit_df.iloc[chosen][['topic_id','subreddit','created_at_utc','sentiment_label','text']].to_dict('records'))
rep_df = pd.DataFrame(representatives)
display(rep_df.head(30))
topics.to_csv(DATA_DIR / 'topic_terms_v2.csv', index=False)
rep_df.to_csv(DATA_DIR / 'topic_representative_posts_v2.csv', index=False)


In [ ]:
# Gán topic toàn bộ parquet theo batch với cùng bước làm sạch.
TOPIC_PATH = DATA_DIR / 'reddit_posts_topics_v2.parquet'
if not TOPIC_PATH.exists():
    pf = pq.ParquetFile(SCORED_PATH)
    writer = None
    try:
        for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=['id','subreddit','created_at_utc','sentiment_label','text']):
            part = batch.to_pandas()
            cleaned = part['text'].map(clean_reddit_text).tolist()
            X = vectorizer.transform(cleaned)
            scores = nmf.transform(X)
            part['topic_id'] = scores.argmax(axis=1).astype('int16')
            part['topic_strength'] = scores.max(axis=1).astype('float32')
            table = pa.Table.from_pandas(part.drop(columns=['text']), preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(TOPIC_PATH, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
print('Topic assignments:', TOPIC_PATH)


In [ ]:
# Tổng hợp topic theo tháng/subreddit và sentiment; xuất phiên bản V2.
tagged = pd.read_parquet(TOPIC_PATH)
tagged['month'] = tagged['created_at_utc'].dt.strftime('%Y-%m')
topic_share = tagged['topic_id'].value_counts(normalize=True).sort_index().mul(100)
print('Topic share trên toàn bộ dữ liệu (%):')
display(topic_share.rename('percent').to_frame())
if topic_share.max() > 50:
    print('CẢNH BÁO: Một topic vẫn chiếm hơn 50% toàn bộ tập; chưa nên diễn giải các tỷ trọng topic như chủ đề ổn định.')
monthly_topic = tagged.groupby(['month','topic_id']).size().rename('posts').reset_index()
monthly_topic.to_csv(DATA_DIR / 'topic_monthly_counts_v2.csv', index=False)
sub_topic = tagged.groupby(['subreddit','topic_id']).size().rename('posts').reset_index()
sub_topic.to_csv(DATA_DIR / 'topic_subreddit_counts_v2.csv', index=False)
topic_sent = tagged.groupby(['topic_id','sentiment_label'], dropna=False).size().rename('posts').reset_index()
topic_sent.to_csv(DATA_DIR / 'topic_sentiment_counts_v2.csv', index=False)
display(monthly_topic.tail(20))
display(sub_topic.sort_values('posts', ascending=False).head(25))
print('Xong. So sánh với kết quả cũ; chỉ đặt tên topic khi từ khóa và bài đại diện khớp nhau.')


## Diễn giải và kiểm tra chất lượng
- `topic_id` là nhãn do mô hình tạo; cần đối chiếu từ khóa và bài đại diện trước khi đặt tên.
- So sánh tỷ trọng topic trong mẫu fit và trên toàn bộ tập. Nếu một topic vẫn chiếm phần lớn, xem đó là topic tổng quát/khó tách và không đưa tỷ trọng này thành kết luận nội dung.
- Các file V2 được lưu riêng để giữ bản cũ. Dùng `topic_terms_v2.csv`, `topic_representative_posts_v2.csv`, `topic_monthly_counts_v2.csv` và `topic_subreddit_counts_v2.csv` để đánh giá lần chạy mới.
- Dataset chỉ bao phủ các subreddit Data Science/AI đã chọn; đây không phải mẫu đại diện cho toàn Reddit.
